
# 09. Class-Weight Sensitivity Analysis

This notebook investigates the effect of giving the 30-day readmission class more importance during model fitting.

The experiment is intentionally narrower than the previous hyperparameter searches:

- the **same cleaned dataset**, 18 predictors, and 60/20/20 model-training/validation/test split are reused;
- the previously selected **non-weight hyperparameters are kept fixed**;
- only the positive-class weighting is changed;
- a common manual weight sequence of **1, 2, 4, 6, 8, and 10** is tested for all four models;
- model-specific automatic/reference settings such as `"balanced"` and `"balanced_subsample"` are also retained where supported;
- five-fold cross-validation on the 60% model-training set measures how class weighting affects **AUPRC, AUROC, and Brier score**;
- for every weight setting, the 20% validation set is used to choose a threshold that achieves at least **80% recall**, then minimises the false-positive rate;
- detailed validation metrics are stored in a single table rather than printing many separate confusion matrices;
- one weight setting per model is selected using **cross-validation AUPRC**, after which its validation-selected threshold is used once on the held-out test set.

## Why select the class weight using cross-validation AUPRC?

Class weight is a model hyperparameter, so it should be selected using the same type of cross-validation logic as the earlier hyperparameter tuning. This keeps the roles of the data splits clear:

1. **Model-training + cross-validation:** choose the class-weight setting.
2. **Validation:** choose the probability threshold for the already selected weight.
3. **Test:** final evaluation only.

The validation results for *all* weights are still reported as a sensitivity analysis. This makes it possible to see whether weighting mainly changes the probability scale, the ranking quality, the false-positive burden, or calibration.


In [ ]:

from pathlib import Path
from copy import deepcopy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    fbeta_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "xgboost is required for this notebook. Install it in the same "
        "environment used for the earlier XGBoost notebook."
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


## 1. Load the cleaned modelling dataset

In [ ]:

PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = parent / "Processed_Dataset" / "diabetic_data_cleaned_stage1.csv"
    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find Processed_Dataset/diabetic_data_cleaned_stage1.csv. "
        "Run this notebook from inside the project folder, or update DATA_PATH."
    )

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Model_Results"
    / "class_weight_sensitivity_analysis"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Data path:", DATA_PATH)
print("Dataset shape:", df.shape)
print("Output directory:", OUTPUT_DIR)


## 2. Reuse the same 18 predictors and binary outcome

In [ ]:

TARGET_COL = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed",
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

model_features = categorical_features + numeric_features

missing_features = [
    feature for feature in model_features + [TARGET_COL]
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        f"Required modelling columns are missing: {missing_features}"
    )

X = df[model_features].copy()
y = df[TARGET_COL].astype(int).copy()

forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr",
}

unexpected_features = forbidden_features.intersection(X.columns)

assert not unexpected_features, (
    f"Unexpected or potentially leaking features found: {unexpected_features}"
)
assert not X.columns.duplicated().any()
assert len(X) == len(y)
assert y.isna().sum() == 0
assert set(y.unique()).issubset({0, 1})

print("X shape:", X.shape)
print("Number of features:", len(model_features))
print("\nTarget counts:")
print(y.value_counts())
print("\nTarget proportions:")
print(y.value_counts(normalize=True))



## 3. Recreate the same 60/20/20 split

The first split reserves 20% as the final test set. The remaining 80% is then split 75/25, giving:

- 60% model-training data;
- 20% threshold-validation data;
- 20% final test data.

All splits are stratified and use `random_state=42`, matching the previous optimisation notebooks.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

X_model_train, X_val, y_model_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    stratify=y_train,
    random_state=42,
)

split_summary = pd.DataFrame({
    "split": [
        "model_train",
        "validation",
        "test",
    ],
    "rows": [
        len(X_model_train),
        len(X_val),
        len(X_test),
    ],
    "positive_count": [
        int(y_model_train.sum()),
        int(y_val.sum()),
        int(y_test.sum()),
    ],
    "positive_rate": [
        y_model_train.mean(),
        y_val.mean(),
        y_test.mean(),
    ],
})

split_summary


## 4. Experiment settings and class-weight groups

In [ ]:

RANDOM_STATE = 42
RECALL_TARGET = 0.80
COMMON_POSITIVE_WEIGHTS = [1, 2, 4, 6, 8, 10]

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Keep this at 1 because Random Forest and XGBoost already parallelise
# internally. This avoids nested parallelism and excessive memory use.
CV_N_JOBS = 1

negative_count = int((y_model_train == 0).sum())
positive_count = int((y_model_train == 1).sum())
IMBALANCE_RATIO = negative_count / positive_count

print("Recall target:", RECALL_TARGET)
print("Common manual positive weights:", COMMON_POSITIVE_WEIGHTS)
print("Model-training negative:positive ratio:", IMBALANCE_RATIO)



### Previously selected non-weight hyperparameters

Only the class-weight parameter will change in this notebook.

- **Regularised Logistic Regression:** \(C=0.01\), L2 regularisation (`l1_ratio=0.0`).
- **Decision Tree:** depth 12, minimum split 100, minimum leaf 400, maximum 127 leaves, `max_features=0.75`, Gini criterion, no cost-complexity pruning.
- **Random Forest:** 500 trees, depth 8, minimum leaf 5, `max_features="log2"`, Gini criterion.
- **XGBoost:** depth 2, `min_child_weight=10`, `gamma=1`, `subsample=0.6`, `colsample_bytree=1.0`, `reg_alpha=1`, `reg_lambda=20`, `max_delta_step=1`, learning rate 0.1, 196 boosting rounds.

The original selected class-weight choices were:

- Logistic Regression: no additional weighting;
- Decision Tree: `"balanced"`;
- Random Forest: no additional weighting;
- XGBoost: `scale_pos_weight=6`.


In [ ]:

LOGISTIC_FIXED_PARAMS = {
    "C": 0.01,
    "l1_ratio": 0.0,
    "solver": "saga",
    "max_iter": 5000,
    "tol": 1e-3,
    "random_state": RANDOM_STATE,
}

DECISION_TREE_FIXED_PARAMS = {
    "min_samples_split": 100,
    "min_samples_leaf": 400,
    "max_leaf_nodes": 127,
    "max_features": 0.75,
    "max_depth": 12,
    "criterion": "gini",
    "ccp_alpha": 0.0,
    "random_state": RANDOM_STATE,
}

RANDOM_FOREST_FIXED_PARAMS = {
    "n_estimators": 500,
    "min_samples_split": 2,
    "min_samples_leaf": 5,
    "max_samples": None,
    "max_features": "log2",
    "max_depth": 8,
    "criterion": "gini",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

XGBOOST_FIXED_PARAMS = {
    "subsample": 0.6,
    "reg_lambda": 20.0,
    "reg_alpha": 1.0,
    "min_child_weight": 10,
    "max_depth": 2,
    "max_delta_step": 1,
    "gamma": 1.0,
    "colsample_bytree": 1.0,
    "learning_rate": 0.1,
    "n_estimators": 196,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "importance_type": "gain",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

selected_parameter_summary = pd.DataFrame([
    {
        "model": "Regularised Logistic Regression",
        "original_selected_weight": "none / ratio 1",
    },
    {
        "model": "Decision Tree",
        "original_selected_weight": "balanced",
    },
    {
        "model": "Random Forest",
        "original_selected_weight": "none / ratio 1",
    },
    {
        "model": "XGBoost",
        "original_selected_weight": "scale_pos_weight=6",
    },
])

selected_parameter_summary


## 5. Preprocessing pipelines

In [ ]:

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

numeric_transformer_scaled = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

numeric_transformer_unscaled = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

logistic_preprocess = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
        (
            "numeric",
            numeric_transformer_scaled,
            numeric_features,
        ),
    ],
    remainder="drop",
)

tree_preprocess = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
        (
            "numeric",
            numeric_transformer_unscaled,
            numeric_features,
        ),
    ],
    remainder="drop",
)

print("Preprocessing objects created.")



## 6. Metric and threshold-selection helper functions

The detailed validation table contains both threshold-independent and threshold-dependent metrics.

**Threshold-independent**
- AUPRC
- AUROC
- Brier score
- log loss
- mean predicted probability
- approximate calibration intercept and slope

**At the validation-selected threshold**
- accuracy and balanced accuracy
- recall and precision
- specificity, false-positive rate, and false-negative rate
- F1 and F2
- TP, TN, FP, and FN
- proportion of patients flagged
- patients flagged per true readmission found


In [ ]:

def calibration_intercept_and_slope(y_true, y_proba):
    """Approximate logistic recalibration intercept and slope.

    Ideal values are intercept = 0 and slope = 1.
    The very large C makes regularisation negligible.
    """
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    clipped = np.clip(y_proba, 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped)).reshape(-1, 1)

    recalibration_model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
    )
    recalibration_model.fit(logits, y_true)

    return (
        float(recalibration_model.intercept_[0]),
        float(recalibration_model.coef_[0, 0]),
    )


def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model",
):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )
    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )
    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )
    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )
    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    calibration_intercept, calibration_slope = (
        calibration_intercept_and_slope(
            y_true,
            y_proba,
        )
    )

    return {
        "model": model_name,
        "threshold": float(threshold),

        "auprc": average_precision_score(y_true, y_proba),
        "auroc": roc_auc_score(y_true, y_proba),
        "brier_score": brier_score_loss(y_true, y_proba),
        "log_loss": log_loss(
            y_true,
            np.clip(y_proba, 1e-15, 1 - 1e-15),
            labels=[0, 1],
        ),
        "observed_prevalence": float(np.mean(y_true)),
        "mean_predicted_probability": float(np.mean(y_proba)),
        "calibration_intercept": calibration_intercept,
        "calibration_slope": calibration_slope,

        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0,
        ),

        "true_positive": int(tp),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "predicted_positive": int(predicted_positive),
        "predicted_negative": int(predicted_negative),
        "predicted_positive_rate": predicted_positive_rate,
        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        ),
    }


def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model",
):
    """Choose the threshold with the lowest FPR subject to recall >= target.

    Tie-breaker: use the highest threshold.
    """
    false_positive_rates, recalls, thresholds = roc_curve(
        y_true,
        y_proba,
        drop_intermediate=False,
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": false_positive_rates,
        "specificity": 1 - false_positive_rates,
    })

    candidate_table = candidate_table[
        np.isfinite(candidate_table["threshold"])
    ].copy()

    eligible_candidates = candidate_table[
        candidate_table["recall"] >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            f"No threshold achieved recall >= {min_recall:.2f}."
        )

    eligible_candidates = (
        eligible_candidates
        .sort_values(
            by=[
                "false_positive_rate",
                "threshold",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        eligible_candidates.iloc[0]["threshold"]
    )

    selected_metrics = evaluate_predictions_from_proba(
        y_true=y_true,
        y_proba=y_proba,
        threshold=selected_threshold,
        model_name=model_name,
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates,
    )


## 7. Define the weight settings

In [ ]:

def sklearn_manual_class_weight(positive_weight):
    if positive_weight == 1:
        # This exactly reproduces an unweighted sklearn classifier.
        return None
    return {0: 1, 1: float(positive_weight)}


logistic_weight_settings = [
    {
        "weight_label": f"manual_{weight}",
        "weight_type": "manual",
        "positive_to_negative_ratio": float(weight),
        "parameter_value": sklearn_manual_class_weight(weight),
    }
    for weight in COMMON_POSITIVE_WEIGHTS
]

logistic_weight_settings.append({
    "weight_label": "balanced",
    "weight_type": "automatic_reference",
    "positive_to_negative_ratio": float(IMBALANCE_RATIO),
    "parameter_value": "balanced",
})


decision_tree_weight_settings = deepcopy(logistic_weight_settings)


random_forest_weight_settings = deepcopy(logistic_weight_settings)
random_forest_weight_settings.append({
    "weight_label": "balanced_subsample",
    "weight_type": "automatic_reference",
    "positive_to_negative_ratio": float(IMBALANCE_RATIO),
    "parameter_value": "balanced_subsample",
})


xgboost_weight_settings = [
    {
        "weight_label": f"manual_{weight}",
        "weight_type": "manual",
        "positive_to_negative_ratio": float(weight),
        "parameter_value": float(weight),
    }
    for weight in COMMON_POSITIVE_WEIGHTS
]

# The original XGBoost search also included the exact class-imbalance ratio.
xgboost_weight_settings.append({
    "weight_label": "imbalance_ratio_reference",
    "weight_type": "automatic_reference",
    "positive_to_negative_ratio": float(IMBALANCE_RATIO),
    "parameter_value": float(IMBALANCE_RATIO),
})


weight_setting_counts = pd.DataFrame({
    "model": [
        "Regularised Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost",
    ],
    "number_of_weight_settings": [
        len(logistic_weight_settings),
        len(decision_tree_weight_settings),
        len(random_forest_weight_settings),
        len(xgboost_weight_settings),
    ],
})

weight_setting_counts


## 8. Model builders with fixed non-weight hyperparameters

In [ ]:

def build_logistic_model(class_weight):
    return Pipeline(
        steps=[
            (
                "preprocess",
                clone(logistic_preprocess),
            ),
            (
                "model",
                LogisticRegression(
                    class_weight=class_weight,
                    **LOGISTIC_FIXED_PARAMS,
                ),
            ),
        ]
    )


def build_decision_tree_model(class_weight):
    return Pipeline(
        steps=[
            (
                "preprocess",
                clone(tree_preprocess),
            ),
            (
                "model",
                DecisionTreeClassifier(
                    class_weight=class_weight,
                    **DECISION_TREE_FIXED_PARAMS,
                ),
            ),
        ]
    )


def build_random_forest_model(class_weight):
    return Pipeline(
        steps=[
            (
                "preprocess",
                clone(tree_preprocess),
            ),
            (
                "model",
                RandomForestClassifier(
                    class_weight=class_weight,
                    **RANDOM_FOREST_FIXED_PARAMS,
                ),
            ),
        ]
    )


def build_xgboost_model(scale_pos_weight):
    return Pipeline(
        steps=[
            (
                "preprocess",
                clone(tree_preprocess),
            ),
            (
                "model",
                XGBClassifier(
                    scale_pos_weight=scale_pos_weight,
                    **XGBOOST_FIXED_PARAMS,
                ),
            ),
        ]
    )


model_experiments = {
    "Regularised Logistic Regression": {
        "builder": build_logistic_model,
        "weight_settings": logistic_weight_settings,
        "original_weight_label": "manual_1",
    },
    "Decision Tree": {
        "builder": build_decision_tree_model,
        "weight_settings": decision_tree_weight_settings,
        "original_weight_label": "balanced",
    },
    "Random Forest": {
        "builder": build_random_forest_model,
        "weight_settings": random_forest_weight_settings,
        "original_weight_label": "manual_1",
    },
    "XGBoost": {
        "builder": build_xgboost_model,
        "weight_settings": xgboost_weight_settings,
        "original_weight_label": "manual_6",
    },
}

print("Model builders created.")



## 9. Cross-validation and validation experiment functions

For each weight setting:

1. run five-fold stratified cross-validation on the model-training data;
2. record mean and standard deviation for AUPRC, AUROC, and Brier score;
3. refit the model on the complete model-training set;
4. obtain validation probabilities;
5. select the validation threshold satisfying recall >= 80% with the lowest FPR;
6. store the full validation statistics in one row.

This means the notebook stays compact even though a large set of statistics is retained.


In [ ]:

CV_SCORING = {
    "auprc": "average_precision",
    "auroc": "roc_auc",
    "neg_brier": "neg_brier_score",
}


def run_cross_validation(model):
    cv_results = cross_validate(
        estimator=model,
        X=X_model_train,
        y=y_model_train,
        scoring=CV_SCORING,
        cv=cross_validation,
        n_jobs=CV_N_JOBS,
        return_train_score=False,
        error_score="raise",
    )

    return {
        "cv_mean_auprc": np.mean(cv_results["test_auprc"]),
        "cv_std_auprc": np.std(cv_results["test_auprc"]),
        "cv_mean_auroc": np.mean(cv_results["test_auroc"]),
        "cv_std_auroc": np.std(cv_results["test_auroc"]),
        "cv_mean_brier": -np.mean(cv_results["test_neg_brier"]),
        "cv_std_brier": np.std(-cv_results["test_neg_brier"]),
    }


def run_one_weight_configuration(
    model_name,
    builder,
    weight_setting,
):
    model = builder(weight_setting["parameter_value"])

    cv_metrics = run_cross_validation(model)

    model.fit(
        X_model_train,
        y_model_train,
    )

    y_val_proba = model.predict_proba(X_val)[:, 1]

    (
        selected_threshold,
        validation_metrics,
        eligible_thresholds,
    ) = choose_threshold_for_minimum_recall(
        y_true=y_val,
        y_proba=y_val_proba,
        min_recall=RECALL_TARGET,
        model_name=model_name,
    )

    result = {
        "model": model_name,
        "weight_label": weight_setting["weight_label"],
        "weight_type": weight_setting["weight_type"],
        "positive_to_negative_ratio": (
            weight_setting["positive_to_negative_ratio"]
        ),
        **cv_metrics,
        **{
            f"val_{key}": value
            for key, value in validation_metrics.items()
            if key != "model"
        },
    }

    return result, model, eligible_thresholds



## 10. Run the sensitivity analysis

**Runtime note:** this is much smaller than the earlier hyperparameter searches, but Random Forest still fits 500 trees in each cross-validation fold. Depending on the computer, the whole cell may take a while. The cell prints only one short progress line per configuration; the detailed results are collected into dataframes.


In [ ]:

all_results = []
fitted_validation_models = {}
eligible_threshold_tables = {}

for model_name, experiment in model_experiments.items():
    print(f"\n=== {model_name} ===")

    for weight_setting in experiment["weight_settings"]:
        label = weight_setting["weight_label"]
        ratio = weight_setting["positive_to_negative_ratio"]

        print(
            f"Running {label} "
            f"(positive:negative ratio ~ {ratio:.3f})"
        )

        (
            result,
            fitted_model,
            eligible_thresholds,
        ) = run_one_weight_configuration(
            model_name=model_name,
            builder=experiment["builder"],
            weight_setting=weight_setting,
        )

        all_results.append(result)
        fitted_validation_models[
            (model_name, label)
        ] = fitted_model
        eligible_threshold_tables[
            (model_name, label)
        ] = eligible_thresholds

class_weight_results = pd.DataFrame(all_results)

print("\nSensitivity analysis complete.")
print("Rows produced:", len(class_weight_results))


## 11. Compact comparison table

In [ ]:

compact_columns = [
    "model",
    "weight_label",
    "weight_type",
    "positive_to_negative_ratio",
    "cv_mean_auprc",
    "cv_std_auprc",
    "cv_mean_auroc",
    "cv_std_auroc",
    "cv_mean_brier",
    "val_threshold",
    "val_auprc",
    "val_auroc",
    "val_brier_score",
    "val_recall",
    "val_precision",
    "val_specificity",
    "val_false_positive_rate",
    "val_f2",
    "val_balanced_accuracy",
    "val_predicted_positive_rate",
]

compact_results = (
    class_weight_results[
        compact_columns
    ]
    .sort_values(
        by=[
            "model",
            "positive_to_negative_ratio",
            "weight_label",
        ]
    )
    .reset_index(drop=True)
)

compact_results



## 12. Full statistics table

Use this table when Dalila wants the complete details. It includes confusion-matrix counts and calibration-related measures, but it is kept as one dataframe instead of producing a separate output block for every model and weight.


In [ ]:

full_columns = [
    "model",
    "weight_label",
    "weight_type",
    "positive_to_negative_ratio",

    "cv_mean_auprc",
    "cv_std_auprc",
    "cv_mean_auroc",
    "cv_std_auroc",
    "cv_mean_brier",
    "cv_std_brier",

    "val_threshold",
    "val_auprc",
    "val_auroc",
    "val_brier_score",
    "val_log_loss",
    "val_observed_prevalence",
    "val_mean_predicted_probability",
    "val_calibration_intercept",
    "val_calibration_slope",

    "val_accuracy",
    "val_balanced_accuracy",
    "val_recall",
    "val_precision",
    "val_specificity",
    "val_false_positive_rate",
    "val_false_negative_rate",
    "val_f1",
    "val_f2",

    "val_true_positive",
    "val_true_negative",
    "val_false_positive",
    "val_false_negative",
    "val_predicted_positive",
    "val_predicted_negative",
    "val_predicted_positive_rate",
    "val_patients_flagged_per_true_readmission_found",
]

full_results = (
    class_weight_results[
        full_columns
    ]
    .sort_values(
        by=[
            "model",
            "positive_to_negative_ratio",
            "weight_label",
        ]
    )
    .reset_index(drop=True)
)

full_results



## 13. Select one class-weight setting per model

The class weight is treated as a **model hyperparameter**, so the selection is based on model-training cross-validation rather than on the test set.

Selection rule:

1. maximise mean five-fold CV AUPRC;
2. if tied, maximise mean CV AUROC;
3. if still tied, minimise mean CV Brier score.

After the weight is selected, its already-computed validation threshold is retained.

The validation FPR/precision/F2 for every other weight remain useful for interpretation, but they do not decide the class-weight hyperparameter.


In [ ]:

selected_weight_rows = []

for model_name in model_experiments:
    model_rows = class_weight_results[
        class_weight_results["model"] == model_name
    ].copy()

    selected_row = (
        model_rows
        .sort_values(
            by=[
                "cv_mean_auprc",
                "cv_mean_auroc",
                "cv_mean_brier",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    selected_weight_rows.append(selected_row)

selected_weights = (
    pd.DataFrame(selected_weight_rows)
    .reset_index(drop=True)
)

selection_columns = [
    "model",
    "weight_label",
    "weight_type",
    "positive_to_negative_ratio",
    "cv_mean_auprc",
    "cv_mean_auroc",
    "cv_mean_brier",
    "val_threshold",
    "val_recall",
    "val_precision",
    "val_false_positive_rate",
    "val_f2",
    "val_brier_score",
    "val_calibration_intercept",
    "val_calibration_slope",
]

selected_weights[selection_columns]



### Compare the newly selected weight with the weight chosen in the earlier broad tuning

This table is particularly useful for answering the supervisor's question about why some models previously selected no additional weighting while others selected stronger positive-class weights.


In [ ]:

original_vs_new_rows = []

for model_name, experiment in model_experiments.items():
    original_label = experiment["original_weight_label"]

    original_row = class_weight_results[
        (class_weight_results["model"] == model_name)
        & (class_weight_results["weight_label"] == original_label)
    ].iloc[0]

    new_row = selected_weights[
        selected_weights["model"] == model_name
    ].iloc[0]

    original_vs_new_rows.append({
        "model": model_name,
        "original_weight": original_label,
        "new_cv_selected_weight": new_row["weight_label"],

        "original_cv_auprc": original_row["cv_mean_auprc"],
        "new_cv_auprc": new_row["cv_mean_auprc"],
        "cv_auprc_change": (
            new_row["cv_mean_auprc"]
            - original_row["cv_mean_auprc"]
        ),

        "original_val_fpr": original_row["val_false_positive_rate"],
        "new_val_fpr": new_row["val_false_positive_rate"],
        "val_fpr_change": (
            new_row["val_false_positive_rate"]
            - original_row["val_false_positive_rate"]
        ),

        "original_val_precision": original_row["val_precision"],
        "new_val_precision": new_row["val_precision"],

        "original_val_brier": original_row["val_brier_score"],
        "new_val_brier": new_row["val_brier_score"],
    })

original_vs_new = pd.DataFrame(original_vs_new_rows)
original_vs_new


## 14. Visualise the class-weight trends

In [ ]:

manual_results = class_weight_results[
    class_weight_results["weight_type"] == "manual"
].copy()

fig, ax = plt.subplots(figsize=(9, 5))

for model_name, group in manual_results.groupby("model"):
    group = group.sort_values("positive_to_negative_ratio")
    ax.plot(
        group["positive_to_negative_ratio"],
        group["cv_mean_auprc"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel("Positive-class weight")
ax.set_ylabel("Mean 5-fold CV AUPRC")
ax.set_title("Cross-validation AUPRC vs positive-class weight")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

auprc_plot_path = OUTPUT_DIR / "class_weight_cv_auprc.png"
plt.savefig(auprc_plot_path, dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))

for model_name, group in manual_results.groupby("model"):
    group = group.sort_values("positive_to_negative_ratio")
    ax.plot(
        group["positive_to_negative_ratio"],
        group["val_false_positive_rate"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel("Positive-class weight")
ax.set_ylabel("Validation false-positive rate")
ax.set_title("FPR at the validation threshold achieving at least 80% recall")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

fpr_plot_path = OUTPUT_DIR / "class_weight_validation_fpr.png"
plt.savefig(fpr_plot_path, dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))

for model_name, group in manual_results.groupby("model"):
    group = group.sort_values("positive_to_negative_ratio")
    ax.plot(
        group["positive_to_negative_ratio"],
        group["val_brier_score"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel("Positive-class weight")
ax.set_ylabel("Validation Brier score")
ax.set_title("Probability quality vs positive-class weight")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

brier_plot_path = OUTPUT_DIR / "class_weight_validation_brier.png"
plt.savefig(brier_plot_path, dpi=200, bbox_inches="tight")
plt.show()



## 15. Final test evaluation of the CV-selected weight only

**Important:** the test set is not used to compare all class weights.

For each model:

1. take the weight selected using five-fold CV AUPRC;
2. use the threshold that was selected for that same weight on the validation set;
3. refit that configuration on the full 60% model-training data;
4. evaluate once on the held-out test set.

Do not change the class weight or threshold after inspecting these results.


In [ ]:

final_test_rows = []
final_test_confusion_matrices = {}

for _, selected_row in selected_weights.iterrows():
    model_name = selected_row["model"]
    weight_label = selected_row["weight_label"]
    selected_threshold = float(selected_row["val_threshold"])

    experiment = model_experiments[model_name]

    weight_setting = next(
        setting
        for setting in experiment["weight_settings"]
        if setting["weight_label"] == weight_label
    )

    final_model = experiment["builder"](
        weight_setting["parameter_value"]
    )

    final_model.fit(
        X_model_train,
        y_model_train,
    )

    y_test_proba = final_model.predict_proba(X_test)[:, 1]

    test_metrics = evaluate_predictions_from_proba(
        y_true=y_test,
        y_proba=y_test_proba,
        threshold=selected_threshold,
        model_name=model_name,
    )

    test_metrics.update({
        "weight_label": weight_label,
        "weight_type": weight_setting["weight_type"],
        "positive_to_negative_ratio": (
            weight_setting["positive_to_negative_ratio"]
        ),
    })

    final_test_rows.append(test_metrics)

    y_test_pred = (
        y_test_proba >= selected_threshold
    ).astype(int)

    cm = confusion_matrix(
        y_test,
        y_test_pred,
        labels=[0, 1],
    )

    final_test_confusion_matrices[model_name] = pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted",
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted",
        ],
    )

final_test_results = pd.DataFrame(final_test_rows)

final_test_columns = [
    "model",
    "weight_label",
    "positive_to_negative_ratio",
    "threshold",
    "auprc",
    "auroc",
    "brier_score",
    "log_loss",
    "calibration_intercept",
    "calibration_slope",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "f1",
    "f2",
    "balanced_accuracy",
    "true_positive",
    "true_negative",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found",
]

final_test_results[final_test_columns]


## 16. Final test confusion matrices

In [ ]:

for model_name, cm_df in final_test_confusion_matrices.items():
    print(f"\n{model_name}")
    display(cm_df)



## 17. Save all outputs

The most useful files for the supervisor discussion are:

- `class_weight_cv_and_validation_full_results.csv`: every statistic for every weight;
- `class_weight_compact_results.csv`: shorter comparison table;
- `class_weight_selected_candidates.csv`: the CV-selected weight for each model;
- `class_weight_original_vs_new.csv`: direct comparison with the weight selected in the earlier broad tuning;
- `class_weight_final_test_results.csv`: final test results for the newly selected configuration only;
- three figures showing AUPRC, validation FPR, and Brier score trends.


In [ ]:

full_results.to_csv(
    OUTPUT_DIR / "class_weight_cv_and_validation_full_results.csv",
    index=False,
)

compact_results.to_csv(
    OUTPUT_DIR / "class_weight_compact_results.csv",
    index=False,
)

selected_weights[selection_columns].to_csv(
    OUTPUT_DIR / "class_weight_selected_candidates.csv",
    index=False,
)

original_vs_new.to_csv(
    OUTPUT_DIR / "class_weight_original_vs_new.csv",
    index=False,
)

final_test_results[final_test_columns].to_csv(
    OUTPUT_DIR / "class_weight_final_test_results.csv",
    index=False,
)

split_summary.to_csv(
    OUTPUT_DIR / "class_weight_split_summary.csv",
    index=False,
)

for model_name, cm_df in final_test_confusion_matrices.items():
    safe_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    cm_df.to_csv(
        OUTPUT_DIR
        / f"class_weight_final_test_confusion_matrix_{safe_name}.csv"
    )

print("Saved outputs to:")
print(OUTPUT_DIR)



## 18. Interpretation guide

When reviewing the results, focus on the following questions:

### A. Did extra class weight improve the model itself?

Look mainly at `cv_mean_auprc` and `cv_mean_auroc`.

- If they improve, the weighting has changed the ranking in a useful way.
- If they remain almost unchanged, weighting is not adding much discrimination.
- If they decline as weight increases, the earlier unweighted choice is understandable.

### B. At the same recall target, did weighting reduce false positives?

Compare:

- `val_false_positive_rate`;
- `val_precision`;
- `val_predicted_positive_rate`;
- `val_f2`.

Because every configuration receives its own validation-selected threshold, this is a fairer comparison than comparing all weights at threshold 0.5.

### C. Did weighting mainly shift the probability scale?

Compare:

- `val_mean_predicted_probability`;
- `val_brier_score`;
- `val_log_loss`;
- `val_calibration_intercept`;
- `val_calibration_slope`.

If AUPRC/AUROC barely change while the predicted probabilities and Brier score change strongly, the main effect of weighting is probably on the probability scale/calibration rather than on patient ranking.

### D. What should be reported?

A concise supervisor summary can use:

1. the compact weight comparison table;
2. the AUPRC-vs-weight plot;
3. the FPR-vs-weight plot;
4. the original-vs-new selected-weight table;
5. the final test table.

The full dataframe remains available if more detail is requested.
